# RQ2 

## Setup

* Follow the instructions on ```Readme.md``` 
* Inside the *pipeline* folder, run ```python -m src.rq2```. This will parse the .json file containing the dataset into  ```bigcodebench_clone_dataset.xml``` file containing all clone pairs (exclusing the original code). 
* Checkout [CloneCognition](https://github.com/pseudoPixels/CloneCognition)
* Paste the ```bigcodebench_clone_dataset.xml``` file inside ```input_clone_pairs/``` on the CloneCognition project.
    * Make sure to follow the CloneCognition instructions on their README
* Run ```python validateClones.py 0.76 input_clone_pairs/ out/```. This will produce a file  ```bigcodebench_clone_dataset.xml.mlValidated```
* The file ```bigcodebench_clone_dataset.xml.mlValidated``` is available in our repo under ```results/RQ2```

In [2]:
from src.config import *
PAIRS = f"../results/RQ2/{DATASET_NAME}_clone_dataset.xml"
RESULTS = f"../results/RQ2/{DATASET_NAME}_clone_dataset.xml.mlValidated"

In [3]:
with open(PAIRS, "r", encoding="utf-8") as f:
    xml_text = f.read()

num_clones = xml_text.count("</clone>")
print("Number of clone pairs:", num_clones)

Number of clone pairs: 56362


## Results

In [4]:
# Summary
from collections import Counter
 
counts = Counter()
total = 0

with open(RESULTS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue  # skip empty lines

        first_field = line.split(",", 1)[0].lower()
        if first_field in {"true", "false"}:
            counts[first_field] += 1
            total += 1

true_count = counts.get("true", 0)
false_count = counts.get("false", 0)

true_pct = (true_count / total * 100) if total > 0 else 0
false_pct = (false_count / total * 100) if total > 0 else 0

print(f"Total entries : {total}")
print(f"Detected clones : {true_count} ({true_pct:.2f}%)")
print(f"Not detected : {false_count} ({false_pct:.2f}%)")


Total entries : 56362
Detected clones : 51 (0.09%)
Not detected : 56311 (99.91%)


In [5]:
# Extract and print entries marked as true
true_entries = []

with open(RESULTS, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        cols = [c.strip() for c in line.split(",")]

        if cols[0].lower() == "true" and len(cols) >= 4:
            true_entries.append(((cols[1], cols[2], cols[3]), (cols[6], cols[7], cols[8])))
 
print("Detected clones:")
for id1, id2 in true_entries:
    print(f"{id1}, {id2}")

print(f"\nTotal detected clones: {len(true_entries)}")

Detected clones:
("BigCodeBench/676_cot deepseek-r1:14b-test 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/676_cot deepseek-r1:14b-complete 1 ['refac_1'", "'refac_3'", "'refac_7']")
("BigCodeBench/293_zero-shot deepseek-r1:14b-test 1 ['refac_1'", "'refac_3'", "'refac_4']"), ("BigCodeBench/293_zero-shot deepseek-r1:14b-test 1 ['refac_2'", "'refac_6'", "'refac_7']")
("BigCodeBench/33_zero-shot gpt-oss:20b-ast 1 ['refac_1'", "'refac_3'", "'refac_4']"), ("BigCodeBench/33_cot gpt-oss:20b-ast 1 ['refac_1'", "'refac_3'", "'refac_4']")
("BigCodeBench/33_cot gemma3:latest-test 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/33_cot llama3.1:latest-test 1 ['refac_1'", "'refac_3'", "'refac_7']")
("BigCodeBench/697_zero-shot gemma3:latest-complete 1 ['refac_2'", "'refac_6'", "'refac_7']"), ("BigCodeBench/697_zero-shot deepseek-r1:14b-complete 1 ['refac_1'", "'refac_4'", "'refac_5']")
("BigCodeBench/275_zero-shot llama3.1:latest-code 1 ['refac_2'", "'refac_4'", "'refac_6']"), (

In [11]:
import pandas as pd
import re
from collections import Counter


with open(RESULTS, "r") as f:
    lines = [l.strip() for l in f if l.strip()]

def parse_side(text):
    entry = re.search(r"(BigCodeBench/\d+)", text)
    strategy = re.search(r"_(cot|zero-shot)", text)
    model = re.search(r"(llama3\.1|deepseek-r1|gpt-oss|gemma3)", text)

    # >>> ADDED: context extraction
    context = re.search(r"-(test|ast|code|complete)\b", text)

    refacs = re.findall(r"refac_\d+", text)

    return {
        "entry": entry.group(1) if entry else None,
        "strategy": strategy.group(1) if strategy else None,
        "model": model.group(1) if model else None,
        "context": context.group(1) if context else None,   # <<< ADDED
        "refacs": refacs,
    }

records = []

for line in lines:
    parts = [p.strip() for p in line.split(",")]

    # Column 0 = TRUE / FALSE
    if parts[0].lower() != "true":
        continue

    # Left and right clone descriptions
    left_text = parts[1]
    right_text = parts[4]

    left = parse_side(left_text)
    right = parse_side(right_text)

    records.append({
        "entry": left["entry"],
        "strategy_left": left["strategy"],
        "strategy_right": right["strategy"],
        "model_left": left["model"],
        "model_right": right["model"],
        "context_left": left["context"],    # <<< ADDED
        "context_right": right["context"],  # <<< ADDED
        "refacs_left": left["refacs"],
        "refacs_right": right["refacs"],
    })

df = pd.DataFrame(records)

entry_counts = df["entry"].value_counts()

strategy_counts = Counter(df["strategy_left"]) + Counter(df["strategy_right"])
model_counts = Counter(df["model_left"]) + Counter(df["model_right"])

# >>> ADDED: context counter
context_counts = Counter(df["context_left"]) + Counter(df["context_right"])

refac_counts = Counter()
for _, r in df.iterrows():
    refac_counts.update(r["refacs_left"])
    refac_counts.update(r["refacs_right"])

strategy_pairs = Counter(
    zip(df["strategy_left"], df["strategy_right"])
)

print("\n=== Most problematic BigCodeBench entries ===")
print(entry_counts.head(10))

print("\n=== Strategy involvement ===")
for k, v in strategy_counts.most_common():
    if k is None:
        continue
    print(f"{k:10s} {v}")

print("\n=== Model involvement ===")
for k, v in model_counts.most_common():
    if k is None:
        continue
    print(f"{k:15s} {v}")

# >>> ADDED: context output
print("\n=== Context involvement ===")
for k, v in context_counts.most_common():
    if k is None:
        continue
    print(f"{k:10s} {v}")

print("\n=== Refactorings involved ===")
for k, v in refac_counts.most_common():
    print(f"{k:8s} {v}")

print("\n=== Strategy pairings ===")
for (s1, s2), v in strategy_pairs.most_common():
    print(f"{s1} vs {s2}: {v}")



=== Most problematic BigCodeBench entries ===
entry
BigCodeBench/911    10
BigCodeBench/546     2
BigCodeBench/122     2
BigCodeBench/4       2
BigCodeBench/33      2
BigCodeBench/667     2
BigCodeBench/697     1
BigCodeBench/293     1
BigCodeBench/676     1
BigCodeBench/275     1
Name: count, dtype: int64

=== Strategy involvement ===
zero-shot  33
cot        18

=== Model involvement ===
deepseek-r1     19
gemma3          15
gpt-oss         10
llama3.1        7

=== Context involvement ===
test       21
ast        11
complete   10
code       9

=== Refactorings involved ===
refac_1  39
refac_2  11
refac_3  1

=== Strategy pairings ===
zero-shot vs None: 33
cot vs None: 18


## Manual validation

In [ ]:
# import json
# import random
# import math
# import pandas as pd
# import numpy as np
# from src.config import *

# random.seed(42)

# # ===== Load dataset =====
# with open(FINAL_DATASET, "r", encoding="utf-8") as f:
#     dataset = json.load(f)

# # ===== Step 1: filter valid entries =====
# all_entries = [
#     entry for entry in dataset
#     if entry.get("original_code") and entry.get("clones")
# ]

# num_entries = len(all_entries)
# sample_entry_size = max(1, int(num_entries * 0.10))

# sampled_entries = random.sample(
#     all_entries,
#     min(sample_entry_size, num_entries)
# )

# print(f"Total entries: {num_entries}")
# print(f"Sampled entries (10%): {len(sampled_entries)}")

# # ===== Step 2 & 3: generate pairs =====
# pairs = []
# included_per_entry = []

# for entry in sampled_entries:
#     entry_id = entry.get("id")
#     original_code = entry.get("original_code")
#     clones = entry.get("clones", [])

#     k = len(clones)
#     if k == 0:
#         continue

#     # sample 30% of clones (rounded up, min 1)
#     clone_sample_size = max(1, math.ceil(0.3 * k))
#     sampled_clones = random.sample(clones, min(clone_sample_size, k))

#     included_per_entry.append(len(sampled_clones))

#     for clone in sampled_clones:
#         clone_code = clone.get("code")
#         clone_id = clone.get("clone_id")

#         codebleu_score = (
#             clone.get("metrics", {})
#                  .get("codebleu", {})
#                  .get("originalcode")
#         )

#         if clone_code:
#             pairs.append(
#                 (
#                     entry_id,
#                     original_code,
#                     clone_code,
#                     1,
#                     "original code",
#                     clone_id,
#                     codebleu_score
#                 )
#             )

# # ===== Create dataframe =====
# df = pd.DataFrame(
#     pairs,
#     columns=[
#         "entry_id",
#         "code_a",
#         "code_b",
#         "label",
#         "source_a",
#         "source_b",
#         "codebleu"
#     ]
# )

# # ===== Save =====
# output_path = "../results/RQ2/clone_inspection_sample.csv"
# df.to_csv(output_path, index=False)

# # ===== Stats =====
# print("\n===== SUMMARY =====")
# print("Saved:", output_path)
# print("Total pairs:", len(df))
# print("Unique entries:", df["entry_id"].nunique())
# print("Labels:", df["label"].unique())

# print("\n===== CLONE SAMPLING STATS =====")
# print(f"Avg clones per entry: {np.mean(included_per_entry):.2f}")
# print(f"Min clones per entry: {np.min(included_per_entry) if included_per_entry else 0}")
# print(f"Max clones per entry: {np.max(included_per_entry) if included_per_entry else 0}")
# print(f"Std dev: {np.std(included_per_entry) if included_per_entry else 0:.2f}")

Total entries: 862
Sampled entries (10%): 86

===== SUMMARY =====
Saved: ../results/RQ2/clone_inspection_sample.csv
Total pairs: 178
Unique entries: 86
Labels: [1]

===== CLONE SAMPLING STATS =====
Avg clones per entry: 2.07
Min clones per entry: 1
Max clones per entry: 5
Std dev: 1.26


### Labeling

In [1]:
!pip install ipywidgets pygments tqdm

In [2]:
import pandas as pd
import random
from IPython.display import display
import ipywidgets as widgets
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
from IPython.core.display import HTML

df = None
to_label_idx = None
state = {"i": 0}
user_col = None
output_path = "../results/RQ2/clone_inspection_sample.csv"


def load_dataset(user_id, start_index=None):
    global df, to_label_idx, user_col, state

    SEED = 42 if user_id == 1 else 10
    user_col = f"manual_label{user_id}"

    df = pd.read_csv(output_path)

    # Ensure label columns exist
    if "manual_label1" not in df.columns:
        df["manual_label1"] = pd.Series(dtype="Int64")
    else:
        df["manual_label1"] = df["manual_label1"].astype("Int64")

    if "manual_label2" not in df.columns:
        df["manual_label2"] = pd.Series(dtype="Int64")
    else:
        df["manual_label2"] = df["manual_label2"].astype("Int64")

    for col in ["manual_label1", "manual_label2"]:
        if col not in df.columns:
            df[col] = pd.Series(dtype="Int64")
        else:
            df[col] = df[col].astype("Int64")

    # Shuffle dataset
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    to_label_idx = df.index.tolist()

    # ===== Correct resume logic =====
    state["i"] = 0
    while state["i"] < len(to_label_idx):
        idx = to_label_idx[state["i"]]
        if pd.isna(df.loc[idx, user_col]):
            break
        state["i"] += 1 



def render_code(code):
    formatter = HtmlFormatter(style="friendly", noclasses=True)
    return HTML(highlight(str(code), PythonLexer(), formatter))


btn_t4 = widgets.Button(description="T4 Clone (4)", button_style="success")
btn_t3 = widgets.Button(description="T3 Clone (3)", button_style="info")
btn_t1 = widgets.Button(description="T1-T2 Clone (1)", button_style="warning")
btn_nonclone = widgets.Button(description="Non-Clone (0)", button_style="danger")

output = widgets.Output()


def show_sample():
    output.clear_output(wait=True)

    while state["i"] < len(to_label_idx):
        idx = to_label_idx[state["i"]]

        # Skip already labeled rows
        if pd.notna(df.loc[idx, user_col]):
            state["i"] += 1
            continue

        row = df.loc[idx]

        labeled = df[user_col].notna().sum()
        total = len(df)

        with output: 
            print(f"Labeled progress: {labeled}/{total}")
            print("=" * 80)
            print(f"User {USER_ID}") 
            print("\nCODE A:\n")
            display(render_code(row["code_a"]))

            print("\nCODE B:\n")
            display(render_code(row["code_b"]))

        return

    with output:
        print("Labeling complete!")

    df.to_csv(output_path, index=False)


def save_label(label):
    global df, user_col

    idx = to_label_idx[state["i"]]
    df.at[idx, user_col] = label
    df.to_csv(output_path, index=False)

    state["i"] += 1
    show_sample()


btn_t4.on_click(lambda _: save_label(4))
btn_t3.on_click(lambda _: save_label(3))
btn_t1.on_click(lambda _: save_label(1))
btn_nonclone.on_click(lambda _: save_label(0))


ui = widgets.VBox([
    widgets.HBox([btn_nonclone, btn_t1, btn_t3, btn_t4]),
    output
])

USER_ID = 1
START_INDEX = 0  # kept for compatibility but not used

display(ui)

load_dataset(user_id=USER_ID, start_index=START_INDEX)
show_sample()

### Analysis

In [4]:
import pandas as pd

# ===== Load dataset =====
path = "../results/RQ2/clone_inspection_sample.csv"


def load_and_analyze(path):
    df = pd.read_csv(path)

    # ===== Add ID column as FIRST column =====
    df = df.reset_index(drop=True)
    if "id" not in df.columns:
        df.insert(0, "id", df.index)
    
    # ===== Ensure nullable integer columns =====
    label_cols = ["label", "manual_label1", "manual_label2"]

    for col in label_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# ===== Save updated CSV AFTER fixing types =====
    df.to_csv(path, index=False)

    # ===== Ensure nullable integer columns =====
    label_cols = ["label", "manual_label1", "manual_label2"]

    for col in label_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    print("DATASET OVERVIEW")
    print(f"Total rows: {len(df)}")

    # ===== Helper mapping =====
    label_names = {
        0: "Non-Clone",
        1: "Type I-II",
        3: "Type III",
        4: "Type IV"
    }

    # ===== Per-annotator summaries =====
    for user_col in ["manual_label1", "manual_label2"]:

        if user_col not in df.columns:
            continue

        valid_df = df.dropna(subset=[user_col]).copy()

        print(f"\nSUMMARY FOR {user_col.upper()}")
        print(f"Labeled samples: {len(valid_df)}")

        counts = (
            valid_df[user_col]
            .value_counts()
            .sort_index()
        )

        print("\nClone type distribution:")

        total = len(valid_df)

        for label_value in [0, 1, 3, 4]:
            count = counts.get(label_value, 0)
            pct = (count / total * 100) if total > 0 else 0

            print(
                f"{label_names[label_value]:<12}: "
                f"{count:>4} ({pct:.2f}%)"
            )

    # ===== Agreement analysis =====
    print("\n" + "=" * 80)
    print("ANNOTATOR AGREEMENT")

    if "manual_label1" in df.columns and "manual_label2" in df.columns:

        agreement_df = df.dropna(
            subset=["manual_label1", "manual_label2"]
        ).copy()

        total_compared = len(agreement_df)

        matching = agreement_df[
            agreement_df["manual_label1"] == agreement_df["manual_label2"]
        ]

        non_matching = agreement_df[
            agreement_df["manual_label1"] != agreement_df["manual_label2"]
        ]

        print(f"Rows labeled by both annotators: {total_compared}")
        print(f"Matching labels: {len(matching)}")
        print(f"Non-matching labels: {len(non_matching)}")

        if total_compared > 0:
            agreement_pct = len(matching) / total_compared * 100
            print(f"Agreement percentage: {agreement_pct:.2f}%")

        # ===== Optional disagreement breakdown =====
        if len(non_matching) > 0:

            print("\nDisagreement label combinations:")

            disagreement_counts = (
                non_matching
                .groupby(["manual_label1", "manual_label2"])
                .size()
                .sort_values(ascending=False)
            )

            for (l1, l2), count in disagreement_counts.items():

                print(
                    f"{label_names.get(l1, l1)} vs "
                    f"{label_names.get(l2, l2)}: {count}"
                )

    else:
        print("manual_label1/manual_label2 columns not found.")

    # ===== Non-Clone ID extraction =====
    print("\n" + "=" * 80)
    print("NON-CLONE CASES (IDs)")
 

    # ---- From annotators ----
    for col in ["manual_label1", "manual_label2"]:
        if col in df.columns:
            ids = df[df[col] == 0]["id"]
            print(f"\nNon-Clone IDs ({col}):")
            print(ids.tolist())

     # ===== Non-Clone ID extraction =====
    print("\n" + "=" * 80)
    print("Type-1--2 CASES (IDs)")
 

    # ---- From annotators ----
    for col in ["manual_label1", "manual_label2"]:
        if col in df.columns:
            ids = df[df[col] == 1]["id"]
            print(f"\nType-1--2 IDs ({col}):")
            print(ids.tolist())


     # ===== Non-Clone ID extraction =====
    print("\n" + "=" * 80)
    print("Type-3 CASES (IDs)")
 

    # ---- From annotators ----
    for col in ["manual_label1", "manual_label2"]:
        if col in df.columns:
            ids = df[df[col] == 3]["id"]
            print(f"\nType-3 IDs ({col}):")
            print(ids.tolist())


    # ---- Agreement (both annotators say Non-Clone) ----
    if "manual_label1" in df.columns and "manual_label2" in df.columns:
        ids = df[
            (df["manual_label1"] == 0) &
            (df["manual_label2"] == 0)
        ]["id"]

        print("\nNon-Clone IDs (agreement):")
        print(ids.tolist())


# ===== Run =====
load_and_analyze(path)

DATASET OVERVIEW
Total rows: 279

SUMMARY FOR MANUAL_LABEL1
Labeled samples: 144

Clone type distribution:
Non-Clone   :    6 (4.17%)
Type I-II   :    8 (5.56%)
Type III    :   25 (17.36%)
Type IV     :  105 (72.92%)

SUMMARY FOR MANUAL_LABEL2
Labeled samples: 0

Clone type distribution:
Non-Clone   :    0 (0.00%)
Type I-II   :    0 (0.00%)
Type III    :    0 (0.00%)
Type IV     :    0 (0.00%)

ANNOTATOR AGREEMENT
Rows labeled by both annotators: 0
Matching labels: 0
Non-matching labels: 0

NON-CLONE CASES (IDs)

Non-Clone IDs (manual_label1):
[4, 14, 18, 50, 68, 156]

Non-Clone IDs (manual_label2):
[]

Type-1--2 CASES (IDs)

Type-1--2 IDs (manual_label1):
[19, 31, 58, 81, 94, 133, 150, 194]

Type-1--2 IDs (manual_label2):
[]

Type-3 CASES (IDs)

Type-3 IDs (manual_label1):
[3, 7, 11, 17, 29, 37, 39, 40, 42, 52, 59, 75, 77, 79, 86, 91, 97, 98, 100, 102, 111, 142, 190, 229, 251]

Type-3 IDs (manual_label2):
[]

Non-Clone IDs (agreement):
[]
